# sample4 - Fine-tuning（LoRA）

**LoRA（Low-Rank Adaptation）** は事前学習済みモデルを少ないパラメータで効率よく Fine-tuning する手法です。  
全パラメータを更新せず、追加する小さな行列だけを学習します。

| ステップ | 内容 |
|----------|------|
| 1 | LoRA の仕組み |
| 2 | ベースモデルの読み込み |
| 3 | LoRA の設定と適用 |
| 4 | データセットの準備 |
| 5 | Fine-tuning の実行 |
| 6 | 推論 |

## 1. LoRA の仕組み

```
通常の Fine-tuning:
  全パラメータ（数十億）を更新 → GPU メモリが大量に必要

LoRA:
  元の重み行列 W は固定
  小さな行列 A・B（ランク r）だけを追加・学習
  W' = W + A × B
  → 学習パラメータが 1% 以下に削減
```

In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
import torch
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("使用デバイス:", device)

使用デバイス: cuda


## 2. ベースモデルの読み込み

軽量な BERT モデルを使用します（初回ダウンロード約270MB）。

In [2]:
model_name = 'bert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 全パラメータ数
total = sum(p.numel() for p in model.parameters())
print(f"全パラメータ数: {total:,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


全パラメータ数: 109,483,778


## 3. LoRA の設定と適用

In [3]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # テキスト分類タスク
    r=8,                          # ランク（小さいほど軽量）
    lora_alpha=32,               # スケーリング係数
    lora_dropout=0.1,
    target_modules=['query', 'value']  # 適用する層
)

peft_model = get_peft_model(model, lora_config)

# 学習可能パラメータの確認
trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"学習可能パラメータ: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
peft_model.print_trainable_parameters()

学習可能パラメータ: 296,450 / 109,483,778 (0.27%)
trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


## 4. データセットの準備

シンプルな感情分析データセットを作成します。

In [4]:
# ポジティブ(1) / ネガティブ(0) のサンプルデータ
train_data = {
    'text': [
        "I love this product!", "Amazing experience!", "Highly recommend!",
        "Great quality and fast delivery.", "Excellent service!",
        "Terrible quality.", "Very disappointed.", "Waste of money.",
        "Poor customer service.", "Would not recommend."
    ],
    'label': [1, 1, 1, 1, 1, 0, 0, 0, 0, 0]
}

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=64)

dataset = Dataset.from_dict(train_data).map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.2)

print("学習データ:", len(dataset['train']))
print("テストデータ:", len(dataset['test']))

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

学習データ: 8
テストデータ: 2


## 5. Fine-tuning の実行

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': (predictions == labels).mean()}

training_args = TrainingArguments(
    output_dir='./lora_output',
    num_train_epochs=5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    eval_strategy='epoch',
    logging_steps=5,
    save_strategy='no'
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    compute_metrics=compute_metrics
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.677670,0.500000
2,No log,0.692497,0.500000
3,0.774152,0.698628,0.500000
4,0.774152,0.706109,0.500000
5,0.782815,0.709686,0.500000


TrainOutput(global_step=10, training_loss=0.778483533859253, metrics={'train_runtime': 0.9776, 'train_samples_per_second': 40.917, 'train_steps_per_second': 10.229, 'total_flos': 1320108748800.0, 'train_loss': 0.778483533859253, 'epoch': 5.0})

## 6. 推論

In [8]:
peft_model.eval()

test_texts = [
    "This is absolutely wonderful!",
    "I hate this so much.",
    "It was okay, nothing special."
]

for text in test_texts:
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # GPU に移動
    with torch.no_grad():
        logits = peft_model(**inputs).logits
    pred = torch.argmax(logits, dim=-1).item()
    label = 'ポジティブ' if pred == 1 else 'ネガティブ'
    print(f"{text}\n → {label}\n")

This is absolutely wonderful!
 → ネガティブ

I hate this so much.
 → ポジティブ

It was okay, nothing special.
 → ポジティブ



# ダウンロードしたモデルを削除

In [10]:
import shutil
import os

hf_cache = os.path.expanduser('~/.cache/huggingface/hub')

models_to_delete = [
    'models--bert-base-uncased',
]

for model_dir in models_to_delete:
    path = os.path.join(hf_cache, model_dir)
    if os.path.exists(path):
        shutil.rmtree(path)
        print(f"削除: {model_dir}")
    else:
        print(f"なし: {model_dir}")

# LoRA 出力の削除
lora_dir = './lora_output'
if os.path.exists(lora_dir):
    shutil.rmtree(lora_dir)
    print(f"削除: {lora_dir}")
else:
    print(f"なし: {lora_dir}")

削除: models--bert-base-uncased
削除: ./lora_output
